# Atividade N3 — Treino do YOLOv8 para Monitoramento de Prateleira (Ruptura de Estoque)

**Objetivo:** treinar (fine-tuning) um **YOLOv8s** para detectar produtos em gôndolas de supermercado, gerando o arquivo `best.pt` usado pelo `main.py` (regra de negócio de ruptura / estoque baixo).

**Dataset:** subconjunto do **SKU-110K** (Goldman et al., CVPR 2019) — fotos reais de prateleiras com centenas de produtos anotados por imagem (classe única: `produto`).
Espelho em Parquet no Hugging Face: `benjamintli/sku110k`.

> Antes de executar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.

In [ ]:
!nvidia-smi
!pip install -q ultralytics pyarrow
import ultralytics; ultralytics.checks()

## 1. Preparação do dataset (Parquet → formato YOLO)

Baixa 4 shards de treino (~1.700 imagens), 1 de validação (~290) e 1 de teste (~420).
Cada imagem é reduzida para no máx. 1024 px e as caixas `[x, y, w, h]` (pixels) são convertidas para o formato YOLO `classe xc yc w h` normalizado.

In [ ]:
import io, urllib.request
from pathlib import Path
import pyarrow.parquet as pq
from PIL import Image

BASE_URL = "https://huggingface.co/datasets/benjamintli/sku110k/resolve/main/data/"
RAIZ = Path("/content/sku110k")
SHARDS = {
    "train": [f"train-{i:05d}-of-00019.parquet" for i in range(4)],
    "val":   ["validation-00000-of-00002.parquet"],
    "test":  ["test-00000-of-00007.parquet"],
}
LADO_MAX = 1024

def converter_shard(parquet, split, prefixo):
    (RAIZ / "images" / split).mkdir(parents=True, exist_ok=True)
    (RAIZ / "labels" / split).mkdir(parents=True, exist_ok=True)
    n_img = n_cx = 0
    for lote in pq.ParquetFile(parquet).iter_batches(batch_size=16, columns=["image", "objects"]):
        for linha in lote.to_pylist():
            img = Image.open(io.BytesIO(linha["image"]["bytes"])).convert("RGB")
            w, h = img.size
            esc = min(1.0, LADO_MAX / max(w, h))
            if esc < 1.0:
                img = img.resize((round(w * esc), round(h * esc)), Image.BILINEAR)
            nome = f"{prefixo}_{n_img:05d}"
            img.save(RAIZ / "images" / split / f"{nome}.jpg", quality=90)
            linhas = []
            for x, y, bw, bh in linha["objects"]["bbox"]:
                x1, y1 = max(0.0, x), max(0.0, y)
                x2, y2 = min(float(w), x + bw), min(float(h), y + bh)
                if x2 - x1 < 1 or y2 - y1 < 1:
                    continue
                linhas.append(f"0 {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
            (RAIZ / "labels" / split / f"{nome}.txt").write_text("\n".join(linhas))
            n_img += 1; n_cx += len(linhas)
    return n_img, n_cx

(RAIZ / "parquet").mkdir(parents=True, exist_ok=True)
for split, nomes in SHARDS.items():
    for k, nome in enumerate(nomes):
        arq = RAIZ / "parquet" / nome
        if not arq.exists():
            print("Baixando", nome); urllib.request.urlretrieve(BASE_URL + nome, arq)
        ni, nc = converter_shard(arq, split, f"{split}{k}")
        print(f"{split}: {nome} -> {ni} imagens, {nc} produtos anotados")

(RAIZ / "sku110k.yaml").write_text(
    f"path: {RAIZ}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: produto\n")
print(open(RAIZ / "sku110k.yaml").read())

## 2. Visualização de uma amostra anotada

In [ ]:
import random, cv2, matplotlib.pyplot as plt
img_path = random.choice(sorted((RAIZ / "images" / "train").glob("*.jpg")))
img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
for l in (RAIZ / "labels" / "train" / (img_path.stem + ".txt")).read_text().splitlines():
    _, xc, yc, bw, bh = map(float, l.split())
    cv2.rectangle(img, (int((xc-bw/2)*w), int((yc-bh/2)*h)), (int((xc+bw/2)*w), int((yc+bh/2)*h)), (0,255,0), 2)
plt.figure(figsize=(12, 9)); plt.imshow(img); plt.axis("off"); plt.title(img_path.name)

## 3. Treinamento (transfer learning a partir do YOLOv8s COCO)

Mesmos hiperparâmetros do treino local (`treinamento/treinar.py`). `max_det=1000` porque cada prateleira tem centenas de produtos.

In [ ]:
from ultralytics import YOLO
modelo = YOLO("yolov8s.pt")
modelo.train(data=str(RAIZ / "sku110k.yaml"), epochs=40, imgsz=640, batch=8, workers=2,
             project="/content/runs", name="sku110k_yolov8s", exist_ok=True,
             max_det=1000, cos_lr=True, close_mosaic=5, patience=15, seed=42, plots=True)

In [ ]:
from IPython.display import Image as IImage, display
run = Path("/content/runs/sku110k_yolov8s")
display(IImage(str(run / "results.png"), width=1000))

## 4. Avaliação no conjunto de teste (imagens nunca vistas)

In [ ]:
best = YOLO(run / "weights" / "best.pt")
m = best.val(data=str(RAIZ / "sku110k.yaml"), split="test", imgsz=640, max_det=1000)
print(f"mAP50={m.box.map50:.3f} | mAP50-95={m.box.map:.3f} | Precisão={m.box.mp:.3f} | Recall={m.box.mr:.3f}")

In [ ]:
# Inferência de exemplo em uma imagem de teste
amostra = sorted((RAIZ / "images" / "test").glob("*.jpg"))[0]
r = best.predict(amostra, imgsz=800, conf=0.35, max_det=1000)[0]
print(len(r.boxes), "produtos detectados")
plt.figure(figsize=(12, 9)); plt.imshow(r.plot(labels=False)[..., ::-1]); plt.axis("off")

## 5. Download do `best.pt`

Copie o arquivo para a pasta `modelo/` do projeto e execute o `main.py`:
```
python main.py --source amostras/
```

In [ ]:
from google.colab import files
files.download(str(run / "weights" / "best.pt"))